# Ingesta de reparto de películas (incremental)

Lee `movie_cast.json` (JSON multilínea) del lote `incremental_append/<p_file_date>` en Bronze, aplica un esquema explícito, estandariza las columnas y agrega `file_date`. Carga la tabla Delta `movie_silver.movie_cast` con `MERGE` (llave `movie_id` + `person_id` + `file_date`), particionada por `file_date`.

### Configuración

Carga las rutas y las funciones comunes, y lee los parámetros `p_environment` y `p_file_date` (fecha del lote).

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16", "file_date")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
from pyspark.sql.functions import col, lit, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

### Lectura del archivo

Primero lee `movie_cast.json` con inferencia para inspeccionarlo y luego lo relee con un esquema explícito.

In [0]:
movie_cast_df = (
    spark.read
        .format("json")
        .option("multiline", "true")
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/movie_cast.json")
    )

In [0]:
movie_cast_schema = StructType([
    StructField("castOrder", IntegerType(), True),
    StructField("characterName", StringType(), True),
    StructField("genderId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("personId", IntegerType(), True)
])
movie_cast_df = (
    spark.read
        .schema(movie_cast_schema)
        .format("json")
        .option("multiline", "true")
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/movie_cast.json")
)

### Transformación de columnas

Conserva `movie_id`, `person_id` y `character_name` (descarta `castOrder` y `genderId`) y agrega `ingestion_date`, `env` y `file_date`.

In [0]:
movie_cast_df = (
    add_ingestion_date(movie_cast_df)
        .select(
            col("movieId").alias("movie_id"),
            col("personId").alias("person_id"),
            col("characterName").alias("character_name"),
            lit(v_env).alias("env"),
            lit(v_file_date).alias("file_date")
        )
)

### Escritura en Silver

Las celdas comentadas conservan estrategias anteriores (borrar partición + `append`). La carga activa es `merge_delta_lake` sobre `movie_silver.movie_cast` con la llave `movie_id` + `person_id` + `file_date`.

In [0]:
#overwrite_partitioned_table("movie_silver", "movie_cast", "file_date", v_file_date)

In [0]:
"""
(
    movie_cast_df
        .write
            .mode("append")
            .partitionBy("file_date")
            .saveAsTable("movie_silver.movie_cast")
)
"""

In [0]:
merge_condition = 'trgt.movie_id = src.movie_id AND trgt.person_id = src.person_id AND trgt.file_date = src.file_date'
merge_delta_lake(input_df = movie_cast_df, schema_name = "movie_silver", table_name = "movie_cast", merge_condition = merge_condition, partition_column = "file_date")

### Fin

Devuelve `Success` al notebook orquestador.

In [0]:
dbutils.notebook.exit("Success")